# Stage 1 — stack-v3-train Curation & Checkpointing
**Kaggle notebook — thin controller only. All logic lives in `scripts/build_sample.py` and `src/curation/`.**

This notebook:
1. Clones the repo
2. Installs the (CPU-only) curation dependencies
3. Sets the HF token from Kaggle Secrets
4. Repairs any checkpoint JSONs left over from the earlier buggy migration
   run (idempotent — safe to re-run)
5. Runs `scripts/build_sample.py` for one `--max-gb`-bounded session
6. Shows the auto-generated filter report

No GPU needed for this stage — see `.claude/data-v1.md` §4. Resuming a later
session picks up automatically from the highest checkpoint already pushed to
the HF dataset repo; nothing to copy-paste between sessions.

In [ ]:
# ── Cell 1: Clone repository at the requested Git state ──────────────────────────
import os
import shutil
import subprocess

GITHUB_REPO = "https://github.com/Rudra-G-23/qwen2.5-coder-0.5b-python-fim.git"

# Set these as needed
BRANCH = "feat/data"  # None -> main
COMMIT = None  # None -> latest commit on BRANCH

REPO_DIR = "/kaggle/working/qwen2.5-coder-0.5b-python-fim"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

clone_cmd = ["git", "clone"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH]
clone_cmd += [GITHUB_REPO, REPO_DIR]
subprocess.run(clone_cmd, check=True)

if COMMIT:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", COMMIT], check=True)

In [ ]:
# ── Cell 2: Install dependencies (CPU-only, no torch/unsloth needed here) ───
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "datasets",
        "pyarrow",
        "huggingface_hub",
        "pyyaml",
    ],
    check=True,
)

In [ ]:
# ── Cell 3: Authenticate to Hugging Face ────────────────────────
# HF_TOKEN is stored as a Kaggle Secret — NEVER hardcode tokens.
# Add it: Kaggle account → Settings → Secrets → Add New Secret
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

In [ ]:
# ── Cell 5: Run one curation session ───────────────────────────
# --max-gb bounds this session's collected (post-filter) bytes — tune to what
# fits comfortably in the 12-hour Kaggle session cap. Re-run this same cell
# next session to resume automatically from the last checkpoint.
import os

os.chdir(REPO_DIR)

MAX_GB = 1

subprocess.run(
    [sys.executable, "scripts/build_sample.py", "--max-gb", str(MAX_GB)],
    check=True,
)

In [ ]:
# ── Cell 6: Show the auto-generated filter report ──────────────────
from IPython.display import Markdown, display

report_path = f"{REPO_DIR}/reports/stage1_filter_report.md"
with open(report_path, encoding="utf-8") as f:
    display(Markdown(f.read()))

In [ ]:
# ── Cell 5: Show the auto-generated filter report ──────────────────
from IPython.display import Markdown, display

report_path = f"{REPO_DIR}/reports/stage1_filter_report.md"
with open(report_path, encoding="utf-8") as f:
    display(Markdown(f.read()))